In [72]:
import os
import weaviate
import weaviate.classes.config as wvc

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

import requests
from bs4 import BeautifulSoup
import tiktoken

In [2]:
load_dotenv(r'.env')

True

In [3]:
def get_content(url):
    response = requests.get(url)

    soup = BeautifulSoup(response.text, 'html.parser')

    for script_or_style in soup(["script", "style", "header", "footer", "nav"]):
        script_or_style.decompose()
        

    text = soup.get_text(separator="\n")

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    clean_text = " ".join(lines)
    return clean_text

In [ ]:
url1 = "https://www.azfamily.com/2024/07/01/evacuation-orders-lifted-boulder-view-fire-near-scottsdale-63-contained/"
url2 = "https://www.kjzz.org/news/2024-06-28/60-forced-to-evacuate-from-boulder-view-fire-in-north-scottsdale"

content1 = get_content(url1)

content2 = get_content(url2)

content = content1 + '\n' + content2


In [6]:
print(content)

Boulder View Fire 95% contained near Scottsdale Skip to content Share Add Us On Google Add as a preferred source on Google SCOTTSDALE, AZ (AZFamily) — The Boulder View Fire, which has burned thousands of acres northwest of Scottsdale, is now 95% contained as of July 4. The Boulder View Fire first sparked on June 27 and has burned 3,711 acres. The fire hasn’t grown in acreage for days. The #BoulderViewFire is now 95% contained at 3,711 acres. This morning the Central West. Zone T3 Incident Management... Posted by Arizona Department of Forestry and Fire Management on Thursday, July 4, 2024 On Sunday, the Maricopa County Department of Emergency Management lifted all “SET” statuses for those near the fire. This means that residents in the evacuation zone are returning to “READY” status as crews made progress on the fire. On Friday, residents in the area from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley were told to evacuate as the fire grew closer to their neighborhood. T

In [10]:
graph_construction_template = '''

-Goal-
Given a text document that is potentially relevant to this activity and a list of entity types, identify all entities of those types from the text and all relationships among the identified entities.

########################################
entities definition:
########################################

- Fire_Event: The primary wildfire event. This is the root node, all other netities should ideally link back to this event.

- Fire_Location: Specific geographic areas or neighborhoods.

- Fire_Cause: The stated reason for the fire's start.

- Status: This is the Status update entity of a fire event, all the status names should contain the time stamp, e.g., Status_June27_1430. Each status contains (all or part of) the following information:
    - Time_Stamp: The timestamp, all the status should contain the timeline information, e.g., ("June 27", "2:30 PM").
    - Burned_Area: The extent of land damaged (acres/sq miles) at a point in time.
    - Evacuation_Detail: Relocation orders and the scale (number of homes/people) at a point in time.
    - Containment_Status: A specific status of the fire's containment at a point in time.
    - Emergency_Response: Firefighting assets and agency measures at a point in time.
    - Public_Sentiment: Community reactions and emotional tone at a point in time.

######################################
relationship definition
######################################
When extracting relationships between identified entities, you MUST use the following predefined relationship labels and logic. Do not use generic labels.

| Relationship Label          | Source Entity | Target Entity        | Extraction Rule                                                                 |
|-----------------------------|---------------|----------------------|---------------------------------------------------------------------------------|
| located_at                  | Fire_Event    | Fire_Location        | Link the primary fire to its geographic origin or currently affected areas.    |
| caused_by                   | Fire_Event    | Fire_Cause           | Link the fire to its stated or suspected ignition source.                      |
| has_status                  | Fire_Event    | Status               | Connect the fire to a specific temporal update or snapshot node.               |
| has_time_stamp              | Status        | Time_Stamp           | Link the status snapshot to a specific date and time (e.g., "June 27, 2:30 PM").|
| has_burned_area             | Status        | Burned_Area          | Link the status snapshot to the acreage damaged at that specific time.         |
| has_evacuation_detail       | Status        | Evacuation_Detail    | Link the status to relocation orders, house counts, or zone levels.            |
| has_containment_status      | Status        | Containment_Status   | Link the status to the current percentage of fire perimeter controlled.        |
| has_emergency_response      | Status        | Emergency_Response   | Link the status to deployed resources like firefighters, aircraft, or crews.   |
| has_public_sentiment        | Status        | Public_Sentiment     | Link the status to the community's emotional tone or documented reactions.     |


Examples:
    - Example 1: 
        Entity (Fire_Event): Boulder View Fire

        Entity (Fire_Location): Northeast of Scottsdale

        Entity (Fire_Cause): Human-caused

        Entity (Status): Status_June27_1430

        Entity (Containment_Status): 60_Percent_Containment

        Entity (3711 ACRES): Burned_Area

        Relationship: Boulder View Fire --(located_at)--> Northeast of Scottsdale
        Relationship: Boulder View Fire --(has_cause)--> Human-caused
        Relationship: Boulder View Fire --(at_time)--> Status_June27_1430
        Relationship: Status_June27_1430 --(has_containment_status) --> 60_Percent_Containment
        Relationship: Status_June27_1430 -- (has_burned_area) --> 3711 ACRES

-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: One of the following types: [{entity_types}]
- entity_description: Comprehensive description of the entity's attributes and activities. You MUST prepend a timestamp to its description in brackets, e.g., [June 27, 2:30 PM] if it has. 
Format each entity as ("entity"{tuple_delimiter}<entity_name>{tuple_delimiter}<entity_type>{tuple_delimiter}<entity_description>)
 
2. From the entities identified in step 1, identify all pairs of (source_entity, relationship, target_entity) that are *clearly related* to each other.

-Relationship Extraction Guidelines-
(1. SNAPSHOT LOGIC: For every new update found in the text (e.g., a morning report vs. an evening report), create a UNIQUE 'Status' entity.
(2. ATTRIBUTE BINDING: All temporal data (Burned_Area, Containment, etc.) must be linked to a 'Status' entity, NOT directly to the 'Fire_Event'. This preserves the chronological evolution.
(3. EXCLUSIVITY: Use only the relationship labels provided in the table above.

For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relationship: One of the followting relationship types: [{relationship_types}]
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other
- relationship_strength: a numeric score indicating strength of the relationship between the source entity and target entity
 Format each relationship as ("relationship"{tuple_delimiter}<source_entity>{tuple_delimiter}<target_entity>{tuple_delimiter}<relationship>{tuple_delimiter}<relationship_description>{tuple_delimiter}<relationship_strength>)
 
3. Return output in English as a single list of all the entities and relationships identified in steps 1 and 2. Use **{record_delimiter}** as the list delimiter.
 
4. When finished, output {completion_delimiter}
 
######################
-Examples-
######################
Example 1:
Entity_types: Fire_Event, Fire_Location, Fire_Cause, Status, Time_Stamp, Burned_Area, Evacuation_Detail, Containment_Status, Emergency_Response, Public_Sentiment
Text:
The Eagle Crest Fire was detected early Monday morning in the Pine Ridge District. Initial investigations suggest the blaze was caused by a lightning strike. As of 9:00 AM, the fire is 0% contained and has scorched 150 acres. Mandatory "GO" orders were issued for the Clear Lake Resort, affecting approximately 20 families. Local residents are reporting high levels of concern due to shifting winds.
######################
Output:
("entity"{tuple_delimiter}EAGLE CREST FIRE{tuple_delimiter}FIRE_EVENT{tuple_delimiter}A fast-moving wildfire event located in the Pine Ridge District)
{record_delimiter}
("entity"{tuple_delimiter}PINE RIDGE DISTRICT{tuple_delimiter}FIRE_LOCATION{tuple_delimiter}The specific forest district where the Eagle Crest Fire originated)
{record_delimiter}
("entity"{tuple_delimiter}LIGHTNING STRIKE{tuple_delimiter}FIRE_CAUSE{tuple_delimiter}The natural ignition source identified for the Eagle Crest Fire)
{record_delimiter}
("entity"{tuple_delimiter}STATUS_MON_0900{tuple_delimiter}STATUS{tuple_delimiter}[Monday, 9:00 AM] Initial assessment of fire growth and evacuation requirements)
{record_delimiter}
("entity"{tuple_delimiter}MONDAY, 9:00 AM{tuple_delimiter}TIME_STAMP{tuple_delimiter}The chronological marker for the first operational period update)
{record_delimiter}
("entity"{tuple_delimiter}150 ACRES{tuple_delimiter}BURNED_AREA{tuple_delimiter}Total acreage affected at the time of the Monday morning update)
{record_delimiter}
("entity"{tuple_delimiter}20 FAMILIES{tuple_delimiter}EVACUATION_DETAIL{tuple_delimiter}Scale of mandatory evacuations at Clear Lake Resort involving 20 households)
{record_delimiter}
("relationship"{tuple_delimiter}EAGLE CREST FIRE{tuple_delimiter}PINE RIDGE DISTRICT{tuple_delimiter}located_at{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}EAGLE CREST FIRE{tuple_delimiter}LIGHTNING STRIKE{tuple_delimiter}caused_by{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}EAGLE CREST FIRE{tuple_delimiter}STATUS_MON_0900{tuple_delimiter}has_status{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_MON_0900{tuple_delimiter}MONDAY, 9:00 AM{tuple_delimiter}has_time_stamp{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_MON_0900{tuple_delimiter}150 ACRES{tuple_delimiter}has_burned_area{tuple_delimiter}9)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_MON_0900{tuple_delimiter}20 FAMILIES{tuple_delimiter}has_evacuation_detail{tuple_delimiter}8)
{completion_delimiter}

Example 2:
Entity_types: Fire_Event, Status, Burned_Area, Containment_Status, Emergency_Response, Public_Sentiment
Text:
Update for Tuesday, 6:00 PM: The Eagle Crest Fire has now reached 800 acres. Firefighting efforts have successfully established lines around 40% of the perimeter. Three Type-1 helicopters have been added to the response. While smoke remains visible, the community is showing signs of cautious optimism as the threat to structures decreases.
######################
Output:
("entity"{tuple_delimiter}EAGLE CREST FIRE{tuple_delimiter}FIRE_EVENT{tuple_delimiter}A fast-moving wildfire event)
{record_delimiter}
("entity"{tuple_delimiter}STATUS_TUE_1800{tuple_delimiter}STATUS{tuple_delimiter}[Tuesday, 6:00 PM] Evening update showing increased containment and additional air support)
{record_delimiter}
("entity"{tuple_delimiter}800 ACRES{tuple_delimiter}BURNED_AREA{tuple_delimiter}Cumulative burned area reported during the Tuesday evening briefing)
{record_delimiter}
("entity"{tuple_delimiter}40%{tuple_delimiter}CONTAINMENT_STATUS{tuple_delimiter}Current percentage of the fire perimeter secured by ground crews)
{record_delimiter}
("entity"{tuple_delimiter}THREE TYPE-1 HELICOPTERS{tuple_delimiter}EMERGENCY_RESPONSE{tuple_delimiter}Specialized aviation assets deployed to assist in water drops)
{record_delimiter}
("entity"{tuple_delimiter}CAUTIOUS OPTIMISM{tuple_delimiter}PUBLIC_SENTIMENT{tuple_delimiter}The prevailing mood of the community following successful containment efforts)
{record_delimiter}
("relationship"{tuple_delimiter}EAGLE CREST FIRE{tuple_delimiter}STATUS_TUE_1800{tuple_delimiter}has_status{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_TUE_1800{tuple_delimiter}800 ACRES{tuple_delimiter}has_burned_area{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_TUE_1800{tuple_delimiter}40%{tuple_delimiter}has_containment_status{tuple_delimiter}10)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_TUE_1800{tuple_delimiter}THREE TYPE-1 HELICOPTERS{tuple_delimiter}has_emergency_response{tuple_delimiter}9)
{record_delimiter}
("relationship"{tuple_delimiter}STATUS_TUE_1800{tuple_delimiter}CAUTIOUS OPTIMISM{tuple_delimiter}has_public_sentiment{tuple_delimiter}8)
{completion_delimiter}

######################
-Real Data-
######################
Entity_types: {entity_types}
Relationship_types: {relationship_types}
Text: {input_text}
######################
Output:
'''


In [100]:
entity_resolution_template = """
There are descriptions of two entities, your role is to determine if there are refering to the exact same entity.

As for the disaster status, if their updated time is not the same, they are identified as two different entities.

Requirements for the output:
- Only output "TRUE“ or "FALSE"
- Don't include your analysis process

##############################
Examples
##############################

Example1:
- entity1 - Name: STATUS_JUNE29_0000, Type: STATUS, Description: [June 29, Early morning] Update describing fire growth and containment status, evacuation trends)
- entity2 - Name: STATUS_JULY4, Type: STATUS, Description: [July 4] The fire is 95% contained with no recent growth in acreage reported)
Output:
"FALSE"

Example2:
- entity1 - Name: NORTH SCOTTSDALE, Type: FIRE_LOCATION, Description: The geographic area in Arizona where the Boulder View Fire originated and is burning)
- entity2 - Name: NORTHWEST OF SCOTTSDALE, Type: FIRE_LOCATION, Description: The geographic area where the Boulder View Fire burned thousands of acres)
Output:
"TRUE"

#############################
Real Data
#############################
These are the two entities:

Entity1: 
{entity1}

Entity2:
{entity2}

Output: 

"""

In [9]:
LLM_client = OpenAI()
model_name = 'gpt-4.1-mini'

In [11]:
entity_types = ["Fire_Event", "Fire_Location", "Fire_Cause", "Burned_Area", "Evacuation_Detail", "Containment_Status", "Emergency_Response", "Public_Sentiment"]
relationship_types = "located_at, caused_by, has_status, has_time_stamp, has_burned_area, has_evacuation_detail, has_containment_status, has_emergency_response, has_public_sentiment"
tuple_delimiter = "::"
record_delimiter = "##"
completion_delimiter = '|COMPLETE|'

In [16]:
graph_construction_prompt = graph_construction_template.format(entity_types=entity_types,
                                                               relationship_types=relationship_types,
                                                               tuple_delimiter=tuple_delimiter,
                                                               record_delimiter=record_delimiter,
                                                               completion_delimiter=completion_delimiter,
                                                               input_text = content1)

In [17]:
response = LLM_client.responses.create(
        model = model_name,
        input = graph_construction_prompt
        )

In [132]:
# print(response.output_text)

In [ ]:
graph_construction_prompt = graph_construction_template.format(entity_types=entity_types,
                                                               relationship_types=relationship_types,
                                                               tuple_delimiter=tuple_delimiter,
                                                               record_delimiter=record_delimiter,
                                                               completion_delimiter=completion_delimiter,
                                                               input_text = content2)
response2 = LLM_client.responses.create(
        model = model_name,
        input = graph_construction_prompt
        )
print(response2.output_text)

In [ ]:
graph_construction_prompt = graph_construction_template.format(entity_types=entity_types,
                                                               relationship_types=relationship_types,
                                                               tuple_delimiter=tuple_delimiter,
                                                               record_delimiter=record_delimiter,
                                                               completion_delimiter=completion_delimiter,
                                                               input_text = content)
all_response = LLM_client.responses.create(
        model = model_name,
        input = graph_construction_prompt
        )
# print(all_response.output_text)

In [46]:
response1 = response.output_text
response2 = response2.output_text

In [89]:
import pandas as pd
def format_graph(texts):
    rows = texts.split("##")
    entities = []
    relationships = []
    for r in rows:
        # embeddings = LLM_client.embeddings.create(input=r, model="text-embedding-3-small", dimensions=1024).data[0].embedding
        if '"entity"' in r:
            r_d = r.split('"entity"')[1].split("::")[1:]
            embeddings = LLM_client.embeddings.create(input=r_d[0], model="text-embedding-3-small", dimensions=1024).data[0].embedding
            clean_r = {"entity_name": r_d[0],
                    "entity_type": r_d[1],
                    "entity_description": r_d[2],
                    "embeddings": embeddings}
            entities.append(clean_r)

        if '"relationship"' in r:
            r_d = r.split('"relationship"')[1].split("::")[1:]
            clean_r = {"source": r_d[0],
                    "target": r_d[1],
                    "relationship": r_d[2],
                    "relationship_description": r_d[3]}
            relationships.append(clean_r)

    entities_df = pd.DataFrame(entities)
    relationships_df = pd.DataFrame(relationships)
    return entities_df, relationships_df

In [90]:
entities_df1, relationships_df1 = format_graph(response1)
entities_df2, relationships_df2 = format_graph(response2)

In [125]:
# entity resolution
def entity_resolution(entities_df1, entities_df2, relationships_df2):
    entities_2_to_merge = []
    for i in range(entities_df2.shape[0]):
        print("################################")

        has_same = False

        entity2 = entities_df2.iloc[i].values.tolist()
        entity2_name = entity2[0]
        entity2_type = entity2[1]
        entity2_description = entity2[2]
        entity2_embeddings = entity2[3]

        entity2_text = f"Name: {entity2[0]}, Type: {entity2[1]}, Description: {entity2[2]}"
        print(entity2_text)

        entity1 = entities_df1[entities_df1["entity_type"]==entity2_type]

        embeddings = entity1['embeddings'].values.tolist()

        sim_matrix = cosine_similarity([entity2_embeddings], embeddings)[0]
        print(sim_matrix)

        for j in range(len(sim_matrix)):
            if sim_matrix[j] > 0.8:
                similar_entity = entity1.iloc[j].values.tolist()
                similar_entity_text = f"Name: {similar_entity[0]}, Type: {similar_entity[1]}, Description: {similar_entity[2]}"
                print(similar_entity_text)
                entity_resolution_prompt = entity_resolution_template.format(entity1=entity2_text, entity2=similar_entity_text)
                resolution_response = LLM_client.responses.create(
                    model = model_name,
                    input = entity_resolution_prompt
                )
                if "TRUE" in resolution_response.output_text:
                    print("They are the same")
                    # modity entities in relationship2
                    relationships_df2_source = relationships_df2['source'].values.tolist()
                    relationships_df2_source = [s.replace(entity2_name, similar_entity[0]) for s in relationships_df2_source]
                    relationships_df2['source'] = relationships_df2_source

                    relationships_df2_target = relationships_df2['target'].values.tolist()
                    relationships_df2_tource = [t.replace(entity2_name, similar_entity[0]) for t in relationships_df2_target]
                    relationships_df2['target'] = relationships_df2_target

                    has_same = True
                    break
        if not has_same:
            entities_2_to_merge.append(dict(entities_df2.iloc[i]))
        
    entities_2_to_merge_df = pd.DataFrame(entities_2_to_merge)

    merged_entities = pd.concat([entities_df1, entities_2_to_merge_df])

    return merged_entities, relationships_df2

# merge relationships
def merge_relationships(relationships_df1, relationships_df2):
    merged_relationships = pd.concat([relationships_df1, relationships_df2])
    cleaned_merged_relationships = merged_relationships.drop_duplicates(subset=['source', 'target', 'relationship'])
    return cleaned_merged_relationships


In [126]:
merged_entities, relationships_df2 = entity_resolution(entities_df1, entities_df2, relationships_df2)
merged_relationships = merge_relationships(relationships_df1, relationships_df2)

################################
Name: BOULDER VIEW FIRE, Type: FIRE_EVENT, Description: A significant wildfire event that began near north Scottsdale, Arizona, causing evacuations and burning thousands of acres)

[1.]
Name: BOULDER VIEW FIRE, Type: FIRE_EVENT, Description: A large wildfire event first ignited on June 27 near Scottsdale, Arizona, burning a total of 3,711 acres and reported 95% contained by July 4)

They are the same
################################
Name: NORTH SCOTTSDALE, Type: FIRE_LOCATION, Description: The geographic area in Arizona where the Boulder View Fire originated and is burning)

[0.85906469 0.35376807 0.43164167]
Name: NORTHWEST OF SCOTTSDALE, Type: FIRE_LOCATION, Description: The geographic area where the Boulder View Fire burned thousands of acres)

They are the same
################################
Name: NEAR TONTO NATIONAL FOREST, Type: FIRE_LOCATION, Description: Area on the edge of which the Boulder View Fire started, about 5 miles outside northern Sc

In [130]:
merged_entities.to_csv(r"merged_entities.csv")

In [131]:
merged_relationships.to_csv(r"merged_relationships.csv")